In [1]:
import torch
from transformers import BertTokenizer, RobertaTokenizer, LongformerTokenizer, AutoTokenizer, AutoModelForMaskedLM
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
import random
from tqdm import tqdm
import os
from helper import dict_lists_to_list_of_dicts

class CustomDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

def pretrain_bert(data_loader, model, optimizer, num_epochs=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.train()
    model.to(device)
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in tqdm(data_loader, desc=f"Epoch {epoch+1}"):
            x = batch.copy()
            del x["output_ids"]
            y = batch["output_ids"].to(device)
            x = {key: val.to(device) for key, val in x.items()}
            outputs = model(**x, labels=y)
            loss = outputs.loss
            total_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            del x
            del y
            torch.cuda.empty_cache()
        print(f"Epoch {epoch+1}, Average Loss: {total_loss / len(data_loader)}")

def get_tokenizer(bert_type:str):
    if "FacebookAI/roberta-base" == bert_type:
        return RobertaTokenizer.from_pretrained(bert_type)
    elif "bert-base-uncased" == bert_type:
        return BertTokenizer.from_pretrained(bert_type)
    elif "allenai/longformer-base-4096" == bert_type:
        return LongformerTokenizer.from_pretrained(bert_type)
    elif "microsoft/codebert-base" == bert_type:
        return AutoTokenizer.from_pretrained(bert_type)
    elif "../weights/tokenizer_weights" == bert_type:
        return AutoTokenizer.from_pretrained(bert_type)
    else:
        raise Exception("bert type unrecognised")

params_files = [f for f in os.listdir("../bert")]
params = []
for file in params_files:
    with open(os.path.join("../bert", file)) as f:
        params.append(f.read())

bert_type = "microsoft/codebert-base"
tokenizer = get_tokenizer("../weights/tokenizer_weights")
model = AutoModelForMaskedLM.from_pretrained(bert_type)

tokenize_params = dict_lists_to_list_of_dicts(tokenizer(params, padding=True, truncation=True, return_tensors='pt'))
for param in tokenize_params:
    param["output_ids"] = param["input_ids"]
    for i in range(param["input_ids"].shape[0]):
        if param["attention_mask"][i] == 0:
            continue
        p = random.random()
        if p > .15:
            p = random.random()
            if p < .8:
                param["input_ids"][i] = tokenizer.mask_token_id
            elif p < .9:
                param["input_ids"][i] = random.randint(0,782)


dataset = CustomDataset(tokenize_params)
data_loader = DataLoader(dataset, batch_size=4, shuffle=True)

optimizer = AdamW(model.parameters(), lr=5e-5)

pretrain_bert(data_loader, model, optimizer)


Some weights of RobertaForMaskedLM were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['lm_head.decoder.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.bias', 'lm_head.bias', 'lm_head.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1: 100%|██████████| 2554/2554 [13:53<00:00,  3.06it/s]


Epoch 1, Average Loss: 0.0867574384003032


Epoch 2: 100%|██████████| 2554/2554 [13:50<00:00,  3.08it/s]


Epoch 2, Average Loss: 0.0076518552612308685


Epoch 3: 100%|██████████| 2554/2554 [14:16<00:00,  2.98it/s]

Epoch 3, Average Loss: 0.006077267035065477


In [2]:
torch.save(model.state_dict(), "../weights/bert/encoder")